# Human grading: run agent on golden cases, grade each trajectory

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from agent.context import config_tag, load_prompt, max_rounds
from agent.judge import JUDGE_PROMPT, parse_json
from agent.knowledge import from_config
from agent.llm import LLM, load_config
from agent.loop import investigate
from agent.schema import CRITERIA, Judgement
from agent.tools import LogTools, tool_purposes
from agent.trajectory import EventLog, Trajectory
from logstore.db import connect

GOLDEN_PATH = ROOT / "labeling" / "golden_cases.yaml"
GRADES_PATH = ROOT / "labeling" / "human_grades.csv"
TRAJ_DIR = ROOT / "labeling" / "trajectories"

config = load_config()
# this run: the knowledge arm. Remove the line to run what config/models.yaml says
config["knowledge"] = True
CONFIG = config_tag(config)
CACHE_TAG = CONFIG

cases = yaml.safe_load(GOLDEN_PATH.read_text(encoding="utf-8"))
print(f"{len(cases)} golden cases: {[c['id'] for c in cases]}")
print(f"config: {CONFIG}  max_rounds: {max_rounds(config)}")

10 golden cases: ['c01', 'c02', 'c03', 'c05', 'c06', 'c07', 'c09', 'c10', 'c12', 'c17']
config: nemotron-3-super-120b-a12b_think200_4r_kb  max_rounds: 4


## Run agent on all cases in parallel

In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def run_one(case):
    conn = connect()
    tools = LogTools(conn, knowledge=from_config(config))
    found = tools.logs.find_by_line(case["log_line"])
    assert len(found) == 1, f"{case['id']}: {len(found)} logs match"
    log = found[0]
    tools.visible = log.ts
    request_id = investigate(log, tools, LLM(config), EventLog(), cache_tag=CACHE_TAG)
    return case["id"], request_id


with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(run_one, c): c["id"] for c in cases}
    results = {}
    for future in as_completed(futures):
        case_id, request_id = future.result()
        results[case_id] = request_id
        t = Trajectory(request_id)
        print(f"{case_id}: {request_id}  status={t.status}  rounds={t.totals['rounds']}  knowledge_calls={sum(e['name'] == 'search_knowledge' for e in t.of_kind('tool_result'))}")

print(f"\ndone: {len(results)} investigations")

c06: fe748c6a3954  status=round_limit  rounds=4  knowledge_calls=0
c01: 2fc5391c0d9b  status=round_limit  rounds=4  knowledge_calls=0
c02: 5d7c8de33bd6  status=round_limit  rounds=4  knowledge_calls=0
c05: 122031488915  status=round_limit  rounds=4  knowledge_calls=0
c03: 750f5c7585e2  status=ok  rounds=4  knowledge_calls=0
c09: 41678cb6b662  status=ok  rounds=3  knowledge_calls=0
c12: 73b013c2607f  status=ok  rounds=3  knowledge_calls=0
c17: 57d17c510ccc  status=round_limit  rounds=4  knowledge_calls=1
c07: 611e42280e4e  status=round_limit  rounds=4  knowledge_calls=1
c10: c19435cfd1e7  status=ok  rounds=2  knowledge_calls=0

done: 10 investigations


## Save trajectories to text files

In [3]:
TRAJ_DIR.mkdir(parents=True, exist_ok=True)

for case_id, request_id in sorted(results.items()):
    path = TRAJ_DIR / f"{case_id}_{CONFIG}__{request_id}.txt"
    path.write_text(Trajectory(request_id).text(), encoding="utf-8")
    print(f"{case_id} -> {path.name}")

print(f"\n{len(results)} trajectories saved to {TRAJ_DIR.relative_to(ROOT)}")

c01 -> c01_nemotron-3-super-120b-a12b_think200_4r_kb__2fc5391c0d9b.txt
c02 -> c02_nemotron-3-super-120b-a12b_think200_4r_kb__5d7c8de33bd6.txt
c03 -> c03_nemotron-3-super-120b-a12b_think200_4r_kb__750f5c7585e2.txt
c05 -> c05_nemotron-3-super-120b-a12b_think200_4r_kb__122031488915.txt
c06 -> c06_nemotron-3-super-120b-a12b_think200_4r_kb__fe748c6a3954.txt
c07 -> c07_nemotron-3-super-120b-a12b_think200_4r_kb__611e42280e4e.txt
c09 -> c09_nemotron-3-super-120b-a12b_think200_4r_kb__41678cb6b662.txt
c10 -> c10_nemotron-3-super-120b-a12b_think200_4r_kb__c19435cfd1e7.txt
c12 -> c12_nemotron-3-super-120b-a12b_think200_4r_kb__73b013c2607f.txt
c17 -> c17_nemotron-3-super-120b-a12b_think200_4r_kb__57d17c510ccc.txt

10 trajectories saved to labeling\trajectories
